# Round 2 — `strips_v4`: thin sharps + the pixels-vs-labels fix

Trains on **`strips_v4`** (40,826 strips / 202 pieces) plus the three promoted **real** pools.
What changed vs `strips_v3` (full rationale: `docs/rung3/round2.md`, numbers: `docs/METRICS.md`):

- **Thin sharps.** All four AEU sharps are redrawn at real-print bar weight. Bravura's bars were 22% too thick and küçük's three bars packed too close, so after the encoder's shrink they fused into a block that *is* a 2-bar koma.
- **The carry pixels-vs-labels fix.** `sigTolerant` was applied to labels but not to the drawing, so **18.8%** of v3's signature-bearing carry strips drew an accidental their label omitted — **2,369 küçük sharps drawn and labelled as nothing**, against 234 labelled correctly. The model was trained to see the glyph and emit nothing, which is exactly its measured failure (48% recall at **100%** precision).
- **+23 küçük-bearing pieces, −5 exam pieces.** v3 rendered our own engraving of five exam pieces; selection now refuses them by SymbTr id.
- **Verified.** `tools/render/verify-labels.ts` compared drawn glyphs against labels for all 40,841 rendered strips: 40,826 exact, 0 label drift. The 15 crop-boundary cases are excluded from the manifest, not trained on.

**No A/B this round.** v3's exam number already exists from Round 1, and real-val cannot arbitrate between corpora (measured 28pp gap to the exam). One recipe — Round 1's winning two-stage arm, **unchanged** — so the exam read is attributable to the corpus fixes rather than to a new schedule.

**The exam is NOT in this zip and is NOT read here.** It is read once, locally, on the final checkpoint.


In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi


In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Copy the data package Drive -> VM disk and unzip (fast local disk for the dataloader).
%%time
!cp /content/drive/MyDrive/tnc/tnc_round2_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_round2_colab.zip
# Expect 40826 strips — if it says 40841 you unzipped a pre-verification build.
!wc -l /content/tnc/data/synthetic/strips_v4/manifest.jsonl
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']),s['stats']['val_strips'],'val strips')"


In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless


In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, the three real pools listed, the exam-disjointness
# line `exam-disjointness OK`, and val loss FALLING.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v4 --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/round2-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2


In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — do this before any long run =====
# Read the s/step below and compute your OWN budget instead of trusting an estimate:
#   hours = (steps * batch) / samples_per_sec / 3600
# Round-1 reference: T4/2-vCPU/workers 2 was ~2.4 s/step @ batch 8 = ~3.3 samples/s (~9 h for a
# 7000-step arm) and was AUGMENTATION-CPU-bound, not GPU-bound.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc   # vCPU count -> the ceiling on --num-workers
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v4 --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

# OPTIONAL A/B on the bottleneck: rerun the same command with --no-augment.
# Much faster => CPU/augmentation-bound (raise --num-workers / bigger runtime).
# Barely faster => GPU-bound (only a bigger GPU or fewer steps helps).


In [ ]:
# ===== STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint that stage 2 specialises.
# Identical to Round 1's winning Arm A stage 1 except for the corpus. TIME: from the calibration cell.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v4 --split data/split_v4.json \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round2-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # ~nproc-2; T4 (2 vCPU) use 2


In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint.
#
# NOTE THE `:9`, NOT Round 1's `:8`. The suffix oversamples each real pool so real is ~1/3 of
# batches. Synthetic grew 33,319 -> 36,057 train strips, so with 2,115 real train strips:
#   :8 -> 31.9%   :9 -> 34.6%   (Round 1 sat at 33.7%)
# :9 is the closer match; leaving it at :8 would quietly weaken the real signal.
#
# Selection caveat carried over from Round 1: oversampled real can overfit fast and `best` is
# picked on a synth-dominated val mix — so evaluate BOTH best and last below.
%cd /content/tnc
!python src/vision/train.py --model /content/drive/MyDrive/tnc/round2-stage1/best \
    --strips-dir data/synthetic/strips_v4 --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota:9 \
    --real-dir data/real/rung3/strips_r1:9 \
    --real-dir data/real/rung3/strips_tup:9 \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round2-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10


In [ ]:
# RESUME after a disconnect: re-run cells 1-4, then this with the SAME flags as the stage you
# were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python src/vision/train.py --strips-dir data/synthetic/strips_v4 --split data/split_v4.json \
    --every-share 0.15 --out-dir /content/drive/MyDrive/tnc/round2-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 --resume


In [ ]:
# ===== REAL-VAL READ — a SANITY CHECK, not a selection number =====
# There is no second arm to choose between this round. Real-val orders candidates; it does NOT
# predict the exam (measured 28pp gap), so treat a good number here as 'nothing broke', not as
# a result. Its one real job: pick between stage-2 `best` and `last`.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir data/real/rung3/strips_nota \
    --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup --split data/split_v4.json

for ck in ['round2-stage2/best', 'round2-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint /content/drive/MyDrive/tnc/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0


## After the run

1. **Download the winning checkpoint** from `MyDrive/tnc/round2-stage2/` to `data/checkpoints/` on the Mac.
2. **Read the exam ONCE**, locally — the exam strips are deliberately not on this VM:
   ```bash
   .venv-ml/bin/python src/vision/eval_omr.py --checkpoint data/checkpoints/<ckpt> \
       --strips-dir data/real/rung3/strips_exam_v2_clean --split none
   ```
   Use `strips_exam_v2_clean` (327 strips) as the honest reference — the full 352 include the
   contaminated pieces. A criterion missed is written up as missed, **never re-rolled on the same exam**.
3. **Read the print-position split too** (`scripts/rung3/score_photo_gold.py`). The microtonal
   sharps are scored almost entirely inside the key signature (exam gold: 32 in-signature vs 1
   inline), so the pooled per-class number cannot tell you whether the fixes worked. On
   `round1-best` küçük read **50% in the signature** where `\bakiyeSharp` read 83% — that gap is
   the thing to watch.
4. **Quote recall AND precision.** `\kucukSharp` precision was 100% on Round 1; recall is what
   these fixes buy, and it can only be bought at precision's expense.
5. **Ship only on a clean pass:** ONNX export → int8 quantize → `onnx_parity.py` (fp32+int8) →
   `make_browser_gate.py` → browser gate → `apps/web/public/models/`.
6. **Attribution caveat to write down:** this round changed the glyph weight AND removed the label
   noise AND added pieces. A move in the sharps cannot be attributed to any one of them.
